# Technical Analysis & Multi-Factor Trading Signals
### From indicators to Buy / Hold / Sell signals with backtesting

**multi-factor technical analysis framework**
real data from **yfinance**. 
Indicators are combined into a scored Buy / Hold / Sell signal that is then backtested against a simple buy-and-hold benchmark.

**Indicators covered:**
- Simple Moving Averages (SMA 30 & 90) — trend following
- RSI 14 — momentum & overbought/oversold conditions
- Bollinger Bands — volatility-adjusted mean reversion
- MACD — trend direction and momentum
- Multi-factor score → categorical signal

**Backtest methodology:**
Long when signal = Buy, flat (cash) when Hold or Sell.

In [23]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import yfinance as yf
import warnings

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "axes.titlesize": 13})
print("libraries imported successfully")

libraries imported successfully


## 1. Data Download

In [25]:
TICKERS = ["NVDA", "AAPL", "MSFT"]
TICKER  = TICKERS[0]          # ticker used for individual indicator sections below
START   = "2021-01-01"

stock  = yf.download(TICKERS, start=START, auto_adjust=True, progress=False)
close  = stock["Close"].dropna()
volume = stock["Volume"]

print(f"Tickers: {', '.join(TICKERS)}")
print(f"{len(close)} observations from {close.index[0].date()} to {close.index[-1].date()}")
print(close.info())
print(close.head(5))

Tickers: NVDA, AAPL, MSFT
1362 observations from 2021-01-04 to 2026-06-05
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1362 entries, 2021-01-04 to 2026-06-05
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AAPL    1362 non-null   float64
 1   MSFT    1362 non-null   float64
 2   NVDA    1362 non-null   float64
dtypes: float64(3)
memory usage: 42.6 KB
None
Ticker            AAPL        MSFT       NVDA
Date                                         
2021-01-04  125.740868  207.956146  13.060796
2021-01-05  127.295479  208.156738  13.350877
2021-01-06  123.010521  202.759369  12.563803
2021-01-07  127.208061  208.529343  13.290368
2021-01-08  128.306000  209.799850  13.223391


## 2. Moving Averages — Trend Detection

The **Golden Cross** (SMA30 crossing above SMA90) is one of the most widely
followed trend-following signals in equity markets.

In [ ]:
sma30 = close[TICKER].rolling(30).mean()
sma90 = close[TICKER].rolling(90).mean()

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True,
                          gridspec_kw={"height_ratios": [3, 1]})

ax = axes[0]
ax.plot(close[TICKER], color="black", linewidth=1.2, label="Close", alpha=0.8)
ax.plot(sma30,  color="steelblue", linewidth=1.8, label="SMA 30")
ax.plot(sma90,  color="coral",     linewidth=1.8, label="SMA 90")

above = sma30 > sma90
for start_i, end_i, val in zip(
    above.index[:-1][above.values[:-1] != above.values[1:]],
    above.index[1:][above.values[:-1]  != above.values[1:]],
    above.values[:-1][above.values[:-1] != above.values[1:]],
):
    ax.axvspan(start_i, end_i, alpha=0.08, color="green" if val else "red")

ax.set_title(f"{TICKER} — Price & Moving Averages")
ax.set_ylabel("Price ($)")
ax.legend()

axes[1].bar(volume[TICKER].index, volume[TICKER] / 1e6, color="steelblue", alpha=0.6, width=1)
axes[1].set_ylabel("Volume (M)")
axes[1].set_xlabel("")
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

plt.tight_layout()
plt.show()

## 3. RSI — Relative Strength Index

$$RSI = 100 - \frac{100}{1 + \dfrac{\text{Avg Gain}_{14}}{\text{Avg Loss}_{14}}}$$

- RSI < 30 → **Oversold** (potential bounce)
- RSI > 70 → **Overbought** (potential reversal)

In [ ]:
def compute_rsi(series: pd.Series, window: int = 14) -> pd.Series:
    delta = series.diff()
    gain  = delta.clip(lower=0).ewm(com=window - 1, adjust=False).mean()
    loss  = (-delta.clip(upper=0)).ewm(com=window - 1, adjust=False).mean()
    rs    = gain / loss.replace(0, np.nan)
    return 100 - 100 / (1 + rs)

rsi14 = compute_rsi(close[TICKER])

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True,
                          gridspec_kw={"height_ratios": [2, 1]})
axes[0].plot(close[TICKER], color="black", linewidth=1.2)
axes[0].set_title(f"{TICKER} — Price & RSI 14")
axes[0].set_ylabel("Price ($)")

axes[1].plot(rsi14, color="purple", linewidth=1.5)
axes[1].axhline(70, color="red",   linestyle="--", linewidth=1.2, label="Overbought (70)")
axes[1].axhline(30, color="green", linestyle="--", linewidth=1.2, label="Oversold (30)")
axes[1].fill_between(rsi14.index, 70, rsi14.clip(upper=100),
                     where=rsi14 > 70, alpha=0.3, color="red")
axes[1].fill_between(rsi14.index, rsi14.clip(lower=0), 30,
                     where=rsi14 < 30, alpha=0.3, color="green")
axes[1].set_ylim(0, 100)
axes[1].set_ylabel("RSI")
axes[1].legend(loc="upper left")

plt.tight_layout()
plt.show()

## 4. Bollinger Bands

Bollinger Bands place bands **±2σ** around a 20-day SMA.
Price touching the lower band signals potential oversold conditions; 
upper band signals overbought.

In [ ]:
bb_mid   = close[TICKER].rolling(20).mean()
bb_std   = close[TICKER].rolling(20).std()
bb_upper = bb_mid + 2 * bb_std
bb_lower = bb_mid - 2 * bb_std
bb_width = (bb_upper - bb_lower) / bb_mid * 100

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True,
                          gridspec_kw={"height_ratios": [3, 1]})
ax = axes[0]
ax.plot(close[TICKER], color="black",     linewidth=1.2, label="Close")
ax.plot(bb_mid,        color="steelblue", linewidth=1.5, linestyle="--", label="SMA 20")
ax.plot(bb_upper,      color="red",       linewidth=1.2, alpha=0.8, label="Upper band (+2σ)")
ax.plot(bb_lower,      color="green",     linewidth=1.2, alpha=0.8, label="Lower band (−2σ)")
ax.fill_between(close[TICKER].index, bb_lower, bb_upper, alpha=0.07, color="steelblue")
ax.set_title(f"{TICKER} — Bollinger Bands (20, 2σ)")
ax.set_ylabel("Price ($)")
ax.legend(ncol=2)

axes[1].plot(bb_width, color="orange", linewidth=1.5)
axes[1].set_ylabel("Band Width (%)")

plt.tight_layout()
plt.show()

## 5. MACD — Moving Average Convergence / Divergence

$$\text{MACD} = \text{EMA}_{12} - \text{EMA}_{26}$$
$$\text{Signal} = \text{EMA}_9(\text{MACD})$$

A MACD crossover above the signal line is a **bullish trigger**.

In [ ]:
ema12 = close[TICKER].ewm(span=12, adjust=False).mean()
ema26 = close[TICKER].ewm(span=26, adjust=False).mean()
macd  = ema12 - ema26
macd_signal = macd.ewm(span=9, adjust=False).mean()
macd_hist   = macd - macd_signal

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True,
                          gridspec_kw={"height_ratios": [2, 1]})
axes[0].plot(close[TICKER], color="black", linewidth=1.2)
axes[0].set_title(f"{TICKER} — MACD")
axes[0].set_ylabel("Price ($)")

axes[1].plot(macd,        color="steelblue", linewidth=1.5, label="MACD")
axes[1].plot(macd_signal, color="coral",     linewidth=1.5, label="Signal")
axes[1].bar(macd_hist.index, macd_hist,
            color=["green" if v >= 0 else "red" for v in macd_hist],
            alpha=0.5, width=1)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_ylabel("MACD")
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Multi-Factor Signal

We aggregate 6 factors into a **discrete score** and map it to a signal:

| Score | Signal |
|-------|--------|
| ≥ 4   | **Buy**  |
| ≤ −3  | **Sell** |
| else  | Hold   |


In [ ]:
def multi_factor_signal(close: pd.Series) -> pd.DataFrame:
    sma30_ = close.rolling(30).mean()
    sma90_ = close.rolling(90).mean()
    rsi_   = compute_rsi(close)
    mom30  = close.pct_change(30) * 100
    mom90  = close.pct_change(90) * 100
    vol90  = close.pct_change().rolling(90).std() * np.sqrt(252) * 100
    draw90 = (close - close.rolling(90).max()) / close.rolling(90).max() * 100

    score = (
        np.sign(close - sma30_)
        + np.sign(close - sma90_)
        + np.sign(sma30_ - sma90_)
        + np.sign(mom30)
        + np.sign(mom90)
        + (rsi_ < 30).astype(float) - (rsi_ > 70).astype(float)
        - (vol90 > 60).astype(float)
        - (draw90 < -15).astype(float)
    )

    signal = score.map(lambda s: "Buy" if s >= 4 else ("Sell" if s <= -3 else "Hold"))
    return pd.DataFrame({"Close": close, "Score": score, "Signal": signal})

signals_all = {t: multi_factor_signal(close[t]).dropna() for t in TICKERS}
colors = {"Buy": "green", "Sell": "red", "Hold": "gold"}

for t in TICKERS:
    signals = signals_all[t]

    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True,
                              gridspec_kw={"height_ratios": [3, 1]})
    axes[0].plot(signals["Close"], color="black", linewidth=1.2, alpha=0.9)
    for sig, col in colors.items():
        mask = signals["Signal"] == sig
        axes[0].scatter(signals.index[mask], signals["Close"][mask],
                        color=col, s=10, label=sig, zorder=3, alpha=0.7)
    axes[0].set_title(f"{t} — Multi-Factor Buy / Hold / Sell Signal")
    axes[0].set_ylabel("Price ($)")
    axes[0].legend()

    axes[1].bar(signals.index, signals["Score"],
                color=[colors[s] for s in signals["Signal"]], alpha=0.7, width=1)
    axes[1].axhline(4,  color="green", linestyle="--", linewidth=1)
    axes[1].axhline(-3, color="red",   linestyle="--", linewidth=1)
    axes[1].set_ylabel("Factor Score")
    axes[1].set_xlabel("")

    plt.tight_layout()
    plt.show()

    dist = signals["Signal"].value_counts()
    print(f"\n{t} signal distribution:")
    print(dist.to_string())

## 7. Simple Backtest: Signal Strategy vs Buy-and-Hold

**Rules:**
- Long (fully invested) when previous day's signal = **Buy**
- Flat (cash, 0% return) otherwise
- No transaction costs (simplifying assumption)

In [ ]:
def metrics(r: pd.Series, label: str):
    ann_r = r.mean() * 252
    ann_v = r.std() * np.sqrt(252)
    sr    = ann_r / ann_v
    mdd   = ((1 + r).cumprod() / (1 + r).cumprod().cummax() - 1).min()
    print(f"  {label:35s}  Return={ann_r:.1%}  Vol={ann_v:.1%}  Sharpe={sr:.2f}  MaxDD={mdd:.1%}")

daily_rets = close.pct_change().dropna()
palette    = ["steelblue", "crimson", "darkorange", "green", "purple"]

print("Backtest Performance")
fig, ax = plt.subplots(figsize=(13, 5))

for i, t in enumerate(TICKERS):
    col       = palette[i % len(palette)]
    daily_ret = daily_rets[t]
    signals   = signals_all[t]
    sig_prev  = signals["Signal"].shift(1).reindex(daily_ret.index).fillna("Hold")
    strat_ret = daily_ret.where(sig_prev == "Buy", 0.0)

    cumul_bh    = (1 + daily_ret).cumprod()
    cumul_strat = (1 + strat_ret).cumprod()

    metrics(daily_ret, f"{t} Buy-and-Hold")
    metrics(strat_ret, f"{t} Signal Strategy")

    ax.plot(cumul_bh.index,    cumul_bh,    linewidth=1.5, linestyle="--", color=col, alpha=0.6, label=f"{t} B&H")
    ax.plot(cumul_strat.index, cumul_strat, linewidth=2,   color=col, label=f"{t} Signal")

ax.set_title("Cumulative Return: Signal Strategy vs Buy-and-Hold")
ax.set_ylabel("Growth of $1")
ax.legend(ncol=2)
plt.tight_layout()
plt.show()